**Intelligent Vision Systems · PyTorch 101 Cookbook** · Dr. Muhammad Kazim · University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/00-pytorch-101-cookbook.ipynb) &nbsp; [Read the cookbook with explanations](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/11_pytorch_101_cookbook.html)

# PyTorch 101 Cookbook: Data → Model → Train & Run

Eight recipes, each a few lines you can run in seconds on a free CPU. Run the cells from top to bottom
(**Shift + Enter**), read each output, then do the **🖐 Try it** change.

## 0 · Set up (10 seconds)

Every recipe below runs on a free Colab CPU in a few seconds. Run this cell first; it imports what all recipes use.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.datasets import load_digits

torch.manual_seed(0)                                     # same numbers every run
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__.split("+")[0], "| device:", device)

If you see device: cuda, everything below runs on the GPU without any change.

## 1 · A Dataset: three methods, nothing more

A Dataset answers two questions: how many samples? (__len__) and what is sample i? (__getitem__). __init__ loads or points to the data once. Here: scikit-learn's 8 × 8 handwritten digits, 1,797 images that ship with the library (no download).

In [ ]:
class DigitsDataset(Dataset):
    """8 x 8 handwritten digits as (image, label) pairs."""

    def __init__(self, transform=None):                   # 1. load once
        d = load_digits()
        self.images = torch.tensor(d.images, dtype=torch.float32) / 16.0   # (N, 8, 8), values 0..1
        self.labels = torch.tensor(d.target, dtype=torch.long)             # (N,)  0..9
        self.transform = transform

    def __len__(self):                                    # 2. how many samples?
        return len(self.labels)

    def __getitem__(self, i):                             # 3. what is sample i?
        x = self.images[i].unsqueeze(0)                   # (1, 8, 8): channels first!
        if self.transform is not None:
            x = self.transform(x)
        return x, self.labels[i]

ds = DigitsDataset()
x, y = ds[0]
print("samples:", len(ds), "| one image:", tuple(x.shape), x.dtype, "| its label:", y.item())

Images are channels first, (C, H, W), and class labels are torch.long: the two details that cause most beginner errors.

**🖐 Try it:** Change __getitem__ so it also returns the index i. What does ds[5] print now? (Change it back afterwards.)

## 2 · Transforms: data augmentation in one line

A transform is any function applied to one sample in __getitem__. Augmentation (small random changes) gives the model more variety for free. torchvision.transforms.v2 works directly on tensors.

In [ ]:
from torchvision.transforms import v2

augment = v2.RandomAffine(degrees=10, translate=(0.1, 0.1))   # small rotation and shift
aug_ds = DigitsDataset(transform=augment)

a, _ = aug_ds[0]
b, _ = aug_ds[0]                                         # same index, new random version
print("shape kept:", tuple(a.shape), "| two versions differ:", not torch.equal(a, b))

Augment the training data only; validation and test data must stay unchanged, or your accuracy is not comparable.

**🖐 Try it:** Set degrees=90. Why is that a bad idea for digits? (Hint: 6 and 9.)

## 3 · Train, validation and test splits

Never measure a model on images it trained on. random_split cuts one dataset into parts; a fixed Generator makes the split the same every run.

In [ ]:
g = torch.Generator().manual_seed(0)
train_ds, val_ds, test_ds = random_split(ds, [0.7, 0.15, 0.15], generator=g)
print("train", len(train_ds), "| validation", len(val_ds), "| test", len(test_ds))

Train to learn, validation to choose settings (epochs, learning rate), test once at the very end.

## 4 · A DataLoader: batches for free

The DataLoader groups samples into batches, shuffles the training data every epoch, and can load in parallel (num_workers). Always look at one batch before training.

In [ ]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)    # shuffle the training data
val_dl = DataLoader(val_ds, batch_size=256)                      # no shuffle needed
test_dl = DataLoader(test_ds, batch_size=256)

xb, yb = next(iter(train_dl))                                     # one batch
print("images:", tuple(xb.shape), "| labels:", tuple(yb.shape), "| first labels:", yb[:8].tolist())

The batch shape is (B, C, H, W) = (64, 1, 8, 8): exactly what a Conv2d expects.

**🖐 Try it:** Set batch_size=1000. How many batches does one epoch have now? (len(train_dl))

## 5 · A model: __init__ = ingredients, forward = recipe

In __init__ you create the layers (the ingredients, with their weights). In forward you say how data flows through them (the recipe). Write the shape after every line as a comment: it is the fastest debugger.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # (B, 1, 8, 8)  -> (B, 16, 8, 8)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # (B, 16, 4, 4) -> (B, 32, 4, 4)
        self.pool = nn.MaxPool2d(2)                                # halves H and W
        self.fc = nn.Linear(32 * 2 * 2, n_classes)                 # (B, 128) -> (B, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))      # (B, 16, 4, 4)
        x = self.pool(F.relu(self.conv2(x)))      # (B, 32, 2, 2)
        return self.fc(x.flatten(1))              # (B, 10): one score (logit) per class

model = TinyCNN().to(device)
out = model(xb.to(device))
print("output:", tuple(out.shape), "| parameters:", sum(p.numel() for p in model.parameters()))

Test a new model with one real batch before training: if the shape is wrong, you find out in one second, not after an hour.

**🖐 Try it:** Remove the second self.pool(...) in forward. Read the error: which line, and which shapes disagree?

## 6 · The training loop: five lines that never change

Every PyTorch project, from this toy to a Transformer, trains with the same five lines inside a loop over the DataLoader. model.train() / model.eval() switch layers such as dropout and BatchNorm between training and testing behaviour.

In [ ]:
def train_one_epoch(model, loader, opt, loss_fn):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()              # 1. forget the old gradients
        logits = model(x)            # 2. forward pass
        loss = loss_fn(logits, y)    # 3. how wrong are we?
        loss.backward()              # 4. gradients for every weight
        opt.step()                   # 5. update the weights
        total += loss.item() * len(x)
    return total / len(loader.dataset)

@torch.no_grad()                     # no gradients needed to evaluate: faster, less memory
def accuracy(model, loader):
    model.eval()
    correct = 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(loader.dataset)

opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()      # takes raw logits and integer labels
for epoch in range(1, 6):
    loss = train_one_epoch(model, train_dl, opt, loss_fn)
    print(f"epoch {epoch} | train loss {loss:.3f} | validation accuracy {accuracy(model, val_dl):.3f}")

The loss goes down and the validation accuracy goes up: the model is learning. That is the whole of "training".

**🖐 Try it:** Delete opt.zero_grad() and rerun from recipe 5. What happens to the loss, and why?

## 7 · Run it: test once, save, load, predict

When you are done choosing settings, measure the test set once. Save only the weights (state_dict); to use them, build the same class and load them.

In [ ]:
print("test accuracy:", round(accuracy(model, test_dl), 3))

torch.save(model.state_dict(), "tiny_cnn.pt")             # weights only, a few KB
model2 = TinyCNN().to(device)
model2.load_state_dict(torch.load("tiny_cnn.pt", map_location=device))
model2.eval()

x, y = test_ds[0]
with torch.no_grad():
    probs = model2(x.unsqueeze(0).to(device)).softmax(dim=1)   # add a batch axis: (1, 1, 8, 8)
print("true:", y.item(), "| predicted:", probs.argmax(1).item(), "| confidence:", round(probs.max().item(), 3))

x.unsqueeze(0) turns one image into a batch of one. Models always expect a batch axis.

## 8 · The sanity check every paper implementer runs

Before a long training run, check that the model can memorise one batch. If the loss does not go to almost zero, there is a bug (shapes, labels, loss, a missing line in the loop) and more data or more epochs will not fix it.

In [ ]:
m = TinyCNN().to(device)
o = torch.optim.Adam(m.parameters(), lr=1e-2)
xb, yb = xb.to(device), yb.to(device)
for step in range(150):
    o.zero_grad()
    loss = loss_fn(m(xb), yb)
    loss.backward()
    o.step()
print("loss on one batch after 150 steps:", round(loss.item(), 4))

This is step 5 of the Lecture 7 recipe ("overfit one batch"). Every Lab 7 model passes it.

## Seven bugs you will meet

| You see | Cause | Fix |
|---|---|---|
| Loss does not go down | forgot opt.zero_grad() or opt.step() | the five lines of recipe 6 |
| expected scalar type Long | labels are float | labels = labels.long() |
| mat1 and mat2 shapes cannot be multiplied | wrong size in the first nn.Linear | print x.shape before it, use that number |
| Expected 4D input | one image without a batch axis | x.unsqueeze(0) |
| Image looks rotated or wrong colours | NumPy/OpenCV image is (H, W, C) | torch.from_numpy(img).permute(2, 0, 1) |
| Validation accuracy jumps around | forgot model.eval() | call it in every evaluation function |
| Expected all tensors on the same device | model on GPU, data on CPU | x.to(device) for every batch |